# Cache and Persist: StorageLevels, Serialization, and Deserialization

**Module**: 06_memory_and_storage  
**Notebook**: 04_cache_and_persist.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Comparing cache() and persist() on DataFrames, StorageLevels, and serialization overhead.
- **Why does this topic exist?**: Proper caching accelerates multi-action pipelines; improper caching destroys performance.
- **What problem does it solve?**: Spilling cached datasets to disk because uncompressed objects consumed too much RAM.
- **Where is it used in real-world Data Engineering?**: Iterative machine learning training loops.

---

## 2. DataFrame cache() = persist(StorageLevel.MEMORY_AND_DISK)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Cache_Persist_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
from pyspark.storagelevel import StorageLevel
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
orders.persist(StorageLevel.MEMORY_AND_DISK)
orders.count()
print(f"Storage Level: {orders.storageLevel}")
orders.unpersist()

---

## Experiment: Timing Cached vs Uncached Second Action

- **1. What are we testing?**: Compare second action duration.
- **2. What do we expect?**: Cached second action finishes 5-10x faster.

In [ ]:
df = orders.filter("quantity > 2").cache()
t0 = time.time(); df.count(); t1 = time.time() - t0
t0 = time.time(); df.count(); t2 = time.time() - t0
print(f"Action 1: {t1:.4f}s, Action 2 (Cached): {t2:.4f}s")
df.unpersist()

**3. What happened?**: Action 2 was drastically faster.  
**4. Why did it happen?**: Read from BlockManager RAM instead of re-reading disk.  
**5. Practical Engineering Lesson**: Cache DataFrames consumed by multiple actions.

---

## Practical Exercises

### Exercise 1 (Beginner): Unpersist Verification
**Problem Statement**: Verify is_cached is False after unpersist.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): DISK_ONLY Level
**Problem Statement**: Persist with StorageLevel.DISK_ONLY.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Storage Footprint Comparison
**Problem Statement**: Compare serialized vs deserialized storage size.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Unpersist Verification):
```python
orders.unpersist(); print('Is cached?:', orders.is_cached)
```

#### Solution 2 (DISK_ONLY Level):
```python
orders.persist(StorageLevel.DISK_ONLY); orders.count(); orders.unpersist()
```

#### Solution 3 (Storage Footprint Comparison):
```python
print('MEMORY_ONLY_SER uses 2-3x less RAM than MEMORY_ONLY')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Rule**: Cache only when reused across 2+ actions.
- **Cleanup**: Always unpersist().

### Essential Technical Terms
- `MEMORY_AND_DISK`: Default DataFrame persist level.
- `unpersist`: Frees memory.

### Common Engineering Mistakes
- **Caching single-use DataFrames**: Wastes storage memory with zero benefit.

### Interview Questions on This Topic
- **Q: What is the default storage level for DataFrames vs RDDs?**
  - *A*: DataFrames default to MEMORY_AND_DISK; RDDs default to MEMORY_ONLY.

### What to Study Next
Open 05_spill_to_disk.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')